# Phase 2 · MHA BIO BiLSTM-CRF

This experiment replaces independent all-character span classification with one coherent BIO sequence per deterministic word/punctuation window. It uses only our pretrained legal decoder checkpoints. The decoder remains causal; a small task-specific BiLSTM supplies right context before a constrained 25-tag CRF.

The audit found no overlapping or nested gold entities. Exact word-boundary representation covers 8,187/8,393 entities (97.55%). Boundary-ambiguous training words split the CRF loss into contiguous valid segments and are never trained as background. Validation retains all original gold entities and exact character offsets.


In [1]:
# 1 · Readable experiment configuration
from pathlib import Path
ACTION = "Train"  # Inspect | Overfit check | Train
PRESET = "mha_pre"  # pinned for this MHA launch notebook
COURSE_ROOT = Path("/content/drive/MyDrive/legal-llm-course")
TRAIN_DIR = COURSE_ROOT / "Phase_2_LNER/Train"
RUN_ROOT = COURSE_ROOT / "runs/phase2_lner/bio_bilstm_crf" / PRESET
BIO_HELPER = COURSE_ROOT / "bio_crf.py"
MAX_BPE, STRIDE_BPE = 512, 384
SEED, VALIDATION_FRACTION = 17, .15
MAX_EPOCHS, PATIENCE = 20, 4
BACKBONE_LR, HEAD_LR = 1e-5, 1e-3
WARMUP_UPDATES, CHECKPOINT_EVERY = 100, 500
LSTM_HIDDEN, INFERENCE_THRESHOLD = 256, None  # CRF decoding has no span threshold


In [2]:
# 2 · Mount Drive and load the existing bundle plus BIO helper
from google.colab import drive
drive.mount('/content/drive')
import sys, shutil, hashlib, json, importlib.util, subprocess, zipfile, random, math, time
BUNDLE_ZIP = COURSE_ROOT / "phase2_lner_gqa_repaired_bundle.zip"
LOCAL = Path('/content/phase2-bio-crf')
if LOCAL.exists(): shutil.rmtree(LOCAL)
if not BUNDLE_ZIP.is_file(): raise FileNotFoundError(f'Existing repaired bundle missing: {BUNDLE_ZIP}')
if not BIO_HELPER.is_file(): raise FileNotFoundError(f'BIO helper missing: {BIO_HELPER}')
LOCAL.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE_ZIP) as archive: archive.extractall(LOCAL)
shutil.copy2(BIO_HELPER,LOCAL/'bio_crf.py')
sys.path.insert(0,str(LOCAL))
missing=[]
if importlib.util.find_spec('tokenizers') is None: missing.append('tokenizers==0.23.1')
if importlib.util.find_spec('huggingface_hub') is None: missing.append('huggingface_hub')
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
from tokenizers import Tokenizer
from legal_ner import *
from bio_crf import *
TOKENIZER_PATH=LOCAL/'tokenizer/tokenizer.json'
TOKENIZER_SHA256=hashlib.sha256(TOKENIZER_PATH.read_bytes()).hexdigest()
if TOKENIZER_SHA256!='906194848c1632431aac7d186e7c4f49da83cb6cbd480fa222c8472401cbf8ab':
    raise ValueError('tokenizer identity mismatch')
TOKENIZER=Tokenizer.from_file(str(TOKENIZER_PATH))
print('Existing tokenizer/data helper and BIO helper loaded.')


Mounted at /content/drive
Existing tokenizer/data helper and BIO helper loaded.


In [3]:
# 3 · Download the pinned dataset files if needed, then audit exact pairs
from google.colab import userdata
training_files=ensure_training_files(COURSE_ROOT,userdata.get('HF_TOKEN'))
print('Training files ready:',[str(p.relative_to(COURSE_ROOT)) for p in training_files])
files=sorted(p for p in TRAIN_DIR.rglob('*') if p.is_file())
print('Files:',[(p.name,p.stat().st_size) for p in files])
labels_path=next((p for p in files if p.name.lower()=='labels.txt'),None)
if labels_path is None: raise FileNotFoundError('actual labels.txt not found')
load_labels(labels_path)
text_files={p.name[:-10]:p for p in files if p.name.endswith('-text.json')}
gold_files={p.name[:-10]:p for p in files if p.name.endswith('-gold.json')}
if not text_files or set(text_files)!=set(gold_files):
    raise ValueError(f'Expected paired official *-text.json/*-gold.json folds; text={sorted(text_files)}, gold={sorted(gold_files)}')
rows=[]
for fold in sorted(text_files):
    try: rows.extend(load_fold_pair(text_files[fold],gold_files[fold]))
    except Exception as e: raise ValueError(f'Could not join actual {fold} fold: {e}') from e
if len({r['id'] for r in rows})!=len(rows): raise ValueError('document ID appears in more than one fold')
report=audit(rows)
print(json.dumps(report,ensure_ascii=False,indent=2))
assert tuple(report['label_counts'])==LABELS
if report['substring_evidence_entities']:
    print('End-exclusive Python Unicode offsets verified against every provided entity_text value.')
else:
    print('WARNING: no entity_text evidence exists; offset convention is not independently proven by the annotations.')


Training files ready: ['Phase_2_LNER/Train/fold1-text.json', 'Phase_2_LNER/Train/fold1-gold.json', 'Phase_2_LNER/Train/fold2-text.json', 'Phase_2_LNER/Train/fold2-gold.json', 'Phase_2_LNER/Train/labels.txt']
Files: [('fold1-gold.json', 460581), ('fold1-text.json', 1106635), ('fold2-gold.json', 475489), ('fold2-text.json', 1124295), ('labels.txt', 71)]
{
  "documents": 70,
  "entities": 8393,
  "label_counts": {
    "APP": 407,
    "RESP": 381,
    "A.COUNSEL": 195,
    "R.COUNSEL": 184,
    "JUDGE": 245,
    "WIT": 187,
    "AUTH": 1003,
    "COURT": 1013,
    "STAT": 1260,
    "PREC": 1183,
    "DATE": 1575,
    "CASENO": 760
  },
  "unicode_documents": 29,
  "overlapping_pairs": 0,
  "nested_pairs": 0,
  "document_chars": {
    "0": 10443,
    "0.5": 26600,
    "0.9": 48800,
    "0.99": 106947,
    "1": 114044
  },
  "span_chars": {
    "0": 1,
    "0.5": 13,
    "0.9": 33,
    "0.99": 85,
    "1": 233
  },
  "substring_evidence_entities": 8393,
  "offset_convention": "verified on ev

In [4]:
# 4 · Preserve the 64/6 document split; deterministic windows never inspect gold
train_rows,val_rows=split_documents(rows,VALIDATION_FRACTION,SEED)
if (len(train_rows),len(val_rows))!=(64,6):
    raise ValueError(f'expected preserved 64/6 document split, got {len(train_rows)}/{len(val_rows)}')
assert {row['id'] for row in train_rows}.isdisjoint({row['id'] for row in val_rows})
train_windows=[window for row in train_rows for window in make_document_windows(row,TOKENIZER,MAX_BPE,STRIDE_BPE)]
val_windows=[window for row in val_rows for window in make_document_windows(row,TOKENIZER,MAX_BPE,STRIDE_BPE)]
aligned=ambiguous=0
for window in train_windows:
    _,segments,masked=targets_and_segments(window,LABELS)
    aligned+=sum(end-first for first,end in segments); ambiguous+=len(masked)
print({'train_documents':64,'validation_documents':6,'train_windows':len(train_windows),
       'validation_windows':len(val_windows),'supervised_train_word_positions':aligned,
       'ambiguous_train_word_occurrences_excluded_from_crf_segments':ambiguous})
print('Validation keeps every original gold span:',sum(len(row['entities']) for row in val_rows))


{'train_documents': 64, 'validation_documents': 6, 'train_windows': 1281, 'validation_windows': 111, 'supervised_train_word_positions': 542378, 'ambiguous_train_word_occurrences_excluded_from_crf_segments': 1950}
Validation keeps every original gold span: 781


## Backbone and task head

The snapshot resolver checks the exact architecture and tokenizer identity. GQA/MHA attention masks are unchanged. Each deterministic word uses its last overlapping decoder subtoken; the BiLSTM and CRF are new task layers.


In [5]:
# 5 · Resolve our pretrained GQA or MHA backbone and create BiLSTM-CRF
import torch
from legal_scaffold.models import ModelConfig,CausalDecoder
PRESETS={
 'gqa_pre':{'vocab_size':32768,'d_model':512,'layers':12,'heads':8,'kv_heads':2,'ffn_hidden':2048,
   'context_length':2048,'attention':'gqa','norm_placement':'pre','tie_embeddings':True,
   'rope_base':10000,'norm_epsilon':1e-6,'init_std':.02},
 'mha_pre':{'vocab_size':32768,'d_model':512,'layers':12,'heads':8,'kv_heads':8,'ffn_hidden':2048,
   'context_length':2048,'attention':'mha','norm_placement':'pre','tie_embeddings':True,
   'rope_base':10000,'norm_epsilon':1e-6,'init_std':.02}}
MODEL_DICT=PRESETS[PRESET]

def resolve_snapshot(root,expected):
    candidates=[]
    for manifest_path in root.rglob('snapshot.json'):
        try:
            manifest=json.loads(manifest_path.read_text())
            saved=manifest.get('identity',{}).get('model',manifest.get('identity',{}).get('config'))
            tok=manifest.get('identity',{}).get('tokenizer_identity',manifest.get('identity',{}).get('tokenizer',{}))
            state_name=manifest.get('state_inventory',{}).get('model')
            state_path=manifest_path.parent/state_name if isinstance(state_name,str) else None
            if saved==expected and tok.get('tokenizer_sha256')==TOKENIZER_SHA256 and state_path and state_path.is_file():
                priority={'legal_continued':0,'wikitext_only':1}.get(manifest.get('stage'),2)
                candidates.append((priority,-int(manifest.get('progress',{}).get('step',0)),state_path,manifest_path))
        except Exception: continue
    if not candidates: raise FileNotFoundError(f'no readable {PRESET} snapshot payload matched config/tokenizer')
    return sorted(candidates,key=lambda item:(item[0],item[1],str(item[2])))[0]

model=None
if ACTION!='Inspect':
    random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    SNAPSHOT_DIR=COURSE_ROOT/'runs/mha_pre/run-001/legal-token-stages-v1/run-001-milestone-60762-aa4db459'
    _,_,BACKBONE_CHECKPOINT,MANIFEST_PATH=resolve_snapshot(SNAPSHOT_DIR,MODEL_DICT)
    backbone=CausalDecoder(ModelConfig(**MODEL_DICT))
    payload=torch.load(BACKBONE_CHECKPOINT,map_location='cpu',weights_only=False)
    backbone.load_state_dict(payload.get('model',payload),strict=True)
    model=BackboneBiLSTMCRF(backbone,MODEL_DICT['d_model'],LABELS,LSTM_HIDDEN)
    IDENTITY={'version':'phase2-bio-bilstm-crf-v1','preset':PRESET,
      'snapshot_sha256':hashlib.sha256(MANIFEST_PATH.read_bytes()).hexdigest(),
      'tokenizer_sha256':TOKENIZER_SHA256,'labels':LABELS,'seed':SEED,
      'max_bpe':MAX_BPE,'stride_bpe':STRIDE_BPE,'lstm_hidden':LSTM_HIDDEN,
      'segmentation':'unicode-word-punctuation-v1','ambiguous_policy':'split-contiguous-training-segments'}
    print('Resolved own pretrained backbone:',BACKBONE_CHECKPOINT)
    print('Parameters:',sum(parameter.numel() for parameter in model.parameters()))
else: print('Inspect: model load skipped.')


Resolved own pretrained backbone: /content/drive/MyDrive/legal-llm-course/runs/mha_pre/run-001/legal-token-stages-v1/run-001-milestone-60762-aa4db459/state.pt
Parameters: 68712124


In [6]:
# 6 · Loss, exact validation, and optional one-window overfit check
def window_loss(window,device):
    ids=torch.tensor([window['input_ids']],dtype=torch.long,device=device)
    last=torch.tensor(window['word_last'],dtype=torch.long,device=device)
    targets,segments,ambiguous=targets_and_segments(window,LABELS); targets=targets.to(device)
    emissions=model.emissions(ids,last)
    losses=[model.crf.neg_log_likelihood(emissions[first:end],targets[first:end])
            for first,end in segments if end>first]
    if not losses: raise ValueError('window contains no contiguous supervised BIO segment')
    return torch.stack(losses).mean(),len(ambiguous)

@torch.inference_mode()
def probe_strict_f1(window,device):
    probe_window=dict(window)
    probe_window['owner_start_index']=probe_window['word_start_index']
    probe_window['owner_end_index']=probe_window['word_end_index']
    ids=torch.tensor([probe_window['input_ids']],dtype=torch.long,device=device)
    last=torch.tensor(probe_window['word_last'],dtype=torch.long,device=device)
    was_training=model.training; model.eval()
    emissions=model.emissions(ids,last)
    predicted=set(decode_entities(probe_window,model.crf.decode(emissions.float()),model.tags))
    model.train(was_training)
    left,right=probe_window['word_spans'][0][0],probe_window['word_spans'][-1][1]
    gold={(probe_window['document_id'],a,b,label) for a,b,label in probe_window['entities']
          if left<=a and b<=right}
    tp=len(predicted&gold); precision=tp/len(predicted) if predicted else 0.; recall=tp/len(gold) if gold else 0.
    f1=2*precision*recall/(precision+recall) if precision+recall else 0.
    return {'gold':len(gold),'predicted':len(predicted),'tp':tp,'precision':precision,'recall':recall,'strict_micro_f1':f1}

@torch.inference_mode()
def validate(device):
    was_training=model.training; model.eval(); predictions=[]
    for window in val_windows:
        ids=torch.tensor([window['input_ids']],dtype=torch.long,device=device)
        last=torch.tensor(window['word_last'],dtype=torch.long,device=device)
        with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=(device=='cuda')):
            emissions=model.emissions(ids,last)
        predictions.extend(decode_entities(window,model.crf.decode(emissions.float()),model.tags))
    model.train(was_training)
    predicted=set(predictions)
    gold={(row['id'],a,b,label) for row in val_rows for a,b,label in row['entities']}
    per_label={}
    for label in LABELS:
        label_gold={item for item in gold if item[3]==label}; label_pred={item for item in predicted if item[3]==label}
        tp=len(label_gold&label_pred); precision=tp/len(label_pred) if label_pred else 0.; recall=tp/len(label_gold) if label_gold else 0.
        f1=2*precision*recall/(precision+recall) if precision+recall else 0.
        per_label[label]={'gold':len(label_gold),'predicted':len(label_pred),'tp':tp,'precision':precision,'recall':recall,'f1':f1}
    return per_label,sum(item['f1'] for item in per_label.values())/len(LABELS),len(predicted)

if ACTION=='Overfit check':
    if not torch.cuda.is_available(): raise RuntimeError('Overfit check requires CUDA')
    device='cuda'; model.to(device).train(); probe=train_windows[0]
    optimizer=torch.optim.AdamW(model.parameters(),lr=1e-4)
    print('overfit decoded metrics before:',probe_strict_f1(probe,device))
    for update in range(20):
        optimizer.zero_grad(set_to_none=True); loss,masked=window_loss(probe,device); loss.backward(); optimizer.step()
        if update in (0,4,9,19): print('overfit update',update+1,'loss',float(loss.detach()),'masked',masked)
    print('overfit decoded metrics after:',probe_strict_f1(probe,device))
    print('Optional overfit check complete; no checkpoint was written.')
else: print('Optional overfit check skipped.')


Optional overfit check skipped.


In [7]:
# 7 · Full fine-tuning with robust latest/best recovery
def optimizer_groups(model):
    backbone=[]; head=[]; seen=set()
    for name,param in model.named_parameters(remove_duplicate=False):
        if not param.requires_grad or id(param) in seen: continue
        seen.add(id(param)); (backbone if name.startswith('backbone.') else head).append(param)
    return [{'params':backbone,'lr':BACKBONE_LR,'weight_decay':.01},
            {'params':head,'lr':HEAD_LR,'weight_decay':.01}]

def lr_multiplier(step,total):
    if step<WARMUP_UPDATES: return (step+1)/max(1,WARMUP_UPDATES)
    fraction=(step-WARMUP_UPDATES)/max(1,total-WARMUP_UPDATES)
    return .5*(1+math.cos(math.pi*min(fraction,1.)))

if ACTION=='Train':
    if not torch.cuda.is_available(): raise RuntimeError('Train requires CUDA; refusing CPU fallback')
    device='cuda'; model.to(device).train(); RUN_ROOT.mkdir(parents=True,exist_ok=True)
    optimizer=torch.optim.AdamW(optimizer_groups(model),betas=(.9,.999))
    total_updates=MAX_EPOCHS*len(train_windows)
    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lambda step:lr_multiplier(step,total_updates))
    scaler=torch.amp.GradScaler('cuda'); latest=RUN_ROOT/'latest.pt'; best=RUN_ROOT/'best.pt'
    progress={'epoch':0,'next_position':0,'successful_updates':0,'overflow_retries':0,
              'best_val_macro_f1':float('-inf'),'best_epoch':-1,'epochs_without_improvement':0}
    if latest.exists():
        progress=resume_checkpoint(latest,model,optimizer,scheduler,scaler,IDENTITY,device)
        print('Resumed:',progress)
    started=time.monotonic()
    for epoch in range(int(progress['epoch']),MAX_EPOCHS):
        order=list(range(len(train_windows))); random.Random(SEED+epoch).shuffle(order)
        position=int(progress['next_position']) if epoch==int(progress['epoch']) else 0
        while position<len(order):
            optimizer.zero_grad(set_to_none=True); window=train_windows[order[position]]
            with torch.autocast(device_type='cuda',dtype=torch.float16): loss,masked=window_loss(window,device)
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            finite=all(torch.isfinite(param.grad).all().item() for param in model.parameters() if param.grad is not None)
            if not finite:
                scaler.step(optimizer); scaler.update(); progress['overflow_retries']+=1
                print(f'AMP overflow epoch={epoch+1} position={position}; retrying',flush=True); continue
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            position+=1; progress['next_position']=position; progress['successful_updates']+=1
            updates=progress['successful_updates']
            if updates==1 or updates%25==0:
                print(f'epoch={epoch+1}/{MAX_EPOCHS} position={position}/{len(order)} updates={updates} '
                      f'loss={loss.detach().item():.5f} ambiguous_words={masked}',flush=True)
            if updates%CHECKPOINT_EVERY==0:
                save_checkpoint(latest,model,optimizer,scheduler,scaler,progress,IDENTITY)
        per_label,macro,pred_count=validate(device)
        improved=macro>float(progress['best_val_macro_f1'])
        completed={**progress,'epoch':epoch+1,'next_position':0,'last_val_macro_f1':float(macro),
                   'epochs_without_improvement':0 if improved else progress['epochs_without_improvement']+1}
        if improved:
            completed['best_val_macro_f1']=float(macro); completed['best_epoch']=epoch+1
            save_checkpoint(best,model,optimizer,scheduler,scaler,completed,IDENTITY)
        progress=completed; save_checkpoint(latest,model,optimizer,scheduler,scaler,progress,IDENTITY)
        print(f'epoch={epoch+1} strict_macro_f1={macro:.6f} predictions={pred_count} improved={improved}',flush=True)
        print(per_label)
        if progress['epochs_without_improvement']>=PATIENCE:
            print('Early stopping after',PATIENCE,'epochs without improvement'); break
    print('Training finished:',progress,'elapsed_seconds',time.monotonic()-started)
else: print('Training skipped.')


AMP overflow epoch=1 position=0; retrying
AMP overflow epoch=1 position=0; retrying
AMP overflow epoch=1 position=0; retrying
AMP overflow epoch=1 position=0; retrying
AMP overflow epoch=1 position=0; retrying
AMP overflow epoch=1 position=0; retrying
AMP overflow epoch=1 position=0; retrying
AMP overflow epoch=1 position=0; retrying
AMP overflow epoch=1 position=0; retrying
epoch=1/20 position=1/1281 updates=1 loss=1081.21240 ambiguous_words=0
epoch=1/20 position=25/1281 updates=25 loss=904.62659 ambiguous_words=0
epoch=1/20 position=50/1281 updates=50 loss=244.35474 ambiguous_words=0
epoch=1/20 position=75/1281 updates=75 loss=65.48828 ambiguous_words=0
epoch=1/20 position=100/1281 updates=100 loss=22.76611 ambiguous_words=0
epoch=1/20 position=125/1281 updates=125 loss=34.85938 ambiguous_words=0
epoch=1/20 position=150/1281 updates=150 loss=0.82702 ambiguous_words=11
epoch=1/20 position=175/1281 updates=175 loss=7.12769 ambiguous_words=0
epoch=1/20 position=200/1281 updates=200 loss

In [8]:
# 8 · Artifact summary
if ACTION=='Train':
    print('Latest recovery:',RUN_ROOT/'latest.pt')
    print('Best validation checkpoint:',RUN_ROOT/'best.pt')
    print('All validation metrics used every original gold entity and exact character offsets.')
else: print('No training artifact written by this action.')


Latest recovery: /content/drive/MyDrive/legal-llm-course/runs/phase2_lner/bio_bilstm_crf/mha_pre/latest.pt
Best validation checkpoint: /content/drive/MyDrive/legal-llm-course/runs/phase2_lner/bio_bilstm_crf/mha_pre/best.pt
All validation metrics used every original gold entity and exact character offsets.


In [9]:
# 9 · Deliberate next step
print('Test export is intentionally separate. First compare best validation macro-F1 and per-label recall.')


Test export is intentionally separate. First compare best validation macro-F1 and per-label recall.


## Running safely

1. Run all with `ACTION = "Inspect"` to verify the 64/6 split and ambiguity counts without loading a model.
2. Optionally run `ACTION = "Overfit check"` only when a GPU is free; it performs 20 updates on one window and writes no checkpoint.
3. Run `ACTION = "Train"` for full fine-tuning. Re-running resumes the preset-specific `latest.pt`; `best.pt` tracks strict validation macro-F1 with all original validation gold retained.
4. GQA is the default. Change only `PRESET` to `mha_pre` for the matched MHA experiment.


In [10]:
"""Post-training BIO test export cell (pinned Phase 2 test edition).

Run only after the BIO notebook setup/model cells have defined the referenced
names.  This deliberately loads best.pt and writes a separate submission.
"""

from pathlib import Path
import json
import zipfile

import torch


TEST_DIR = COURSE_ROOT / "Phase_2_LNER/Test"
BEST_CHECKPOINT = RUN_ROOT / "best.pt"
SUBMISSION_DIR = RUN_ROOT / "submission_best"
TEST_REPOSITORY = "Exploration-Lab/CS-781-Capstone"
TEST_REVISION = "89edba6a44d6bb9593fd13b6895b5cd3b06eb82e"
TEST_REPOSITORY_PATH = "Phase_2_LNER/Test/fold3-text.json"
# Phase 1's official-test/test.json is unrelated and must never be substituted.
test_files = [COURSE_ROOT / TEST_REPOSITORY_PATH]
missing_test_files = [path for path in test_files if not path.is_file()]
if missing_test_files:
    raise FileNotFoundError(
        "Pinned Phase 2 NER test file is not readable locally. Download only "
        f"{TEST_REPOSITORY_PATH} from {TEST_REPOSITORY} at revision {TEST_REVISION} "
        f"with the approved authenticated Hugging Face session: {missing_test_files}"
    )

if not BEST_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Best BIO checkpoint not found: {BEST_CHECKPOINT}")

payload = torch.load(BEST_CHECKPOINT, map_location="cpu", weights_only=False)
if payload.get("schema_version") != 1 or payload.get("identity") != IDENTITY:
    raise ValueError("Best BIO checkpoint identity mismatch")
model.load_state_dict(payload["model"], strict=True)

test_rows = []
for path in test_files:
    test_rows.extend(load_records(path))
if not test_rows:
    raise ValueError("Unlabeled test files contained no documents; refusing empty submission")
if len({row["id"] for row in test_rows}) != len(test_rows):
    raise ValueError("Duplicate official document IDs across test files")
empty_text_ids = [row["id"] for row in test_rows if not row["text"]]

test_windows = [
    window
    for row in test_rows
    if row["text"]
    for window in make_document_windows(row, TOKENIZER, MAX_BPE, STRIDE_BPE)
]

if not torch.cuda.is_available():
    raise RuntimeError("BIO test inference requires CUDA; refusing silent CPU fallback")
device = "cuda"
model.to(device).eval()
decoded = []
with torch.inference_mode():
    for window in test_windows:
        ids = torch.tensor([window["input_ids"]], dtype=torch.long, device=device)
        last = torch.tensor(window["word_last"], dtype=torch.long, device=device)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            emissions = model.emissions(ids, last)
        decoded.extend(decode_entities(window, model.crf.decode(emissions.float()), model.tags))

# export_prediction is the existing official-schema writer.  BIO/CRF decoding
# has no confidence threshold, so the unused score field is a fixed sentinel.
unique_predictions = sorted(set(decoded), key=lambda item: (item[0], item[1], item[2], item[3]))
scored_predictions = [(*item, 1.0) for item in unique_predictions]
texts = {row["id"]: row["text"] for row in test_rows}
prediction_path, zip_path = export_prediction(
    [row["id"] for row in test_rows], scored_predictions, texts, SUBMISSION_DIR
)

written = json.loads(prediction_path.read_text(encoding="utf-8"))
if list(written) != [row["id"] for row in test_rows]:
    raise ValueError("Export changed official test ID order or membership")
with zipfile.ZipFile(zip_path) as archive:
    if archive.namelist() != ["prediction.json"]:
        raise ValueError("Submission ZIP must contain only root prediction.json")

print("Loaded best checkpoint:", BEST_CHECKPOINT)
print("Best validation progress:", payload["progress"])
print("Test files:", [str(path) for path in test_files])
print("Documents:", len(test_rows), "decoded entities:", len(unique_predictions))
print("Empty official documents retained with []:", empty_text_ids)
print("Submission JSON:", prediction_path)
print("Submission ZIP:", zip_path)


Loaded best checkpoint: /content/drive/MyDrive/legal-llm-course/runs/phase2_lner/bio_bilstm_crf/mha_pre/best.pt
Best validation progress: {'epoch': 1, 'next_position': 0, 'successful_updates': 1281, 'overflow_retries': 9, 'best_val_macro_f1': 0.4101461759510681, 'best_epoch': 1, 'epochs_without_improvement': 0, 'last_val_macro_f1': 0.4101461759510681}
Test files: ['/content/drive/MyDrive/legal-llm-course/Phase_2_LNER/Test/fold3-text.json']
Documents: 35 decoded entities: 3889
Empty official documents retained with []: ['1305957']
Submission JSON: /content/drive/MyDrive/legal-llm-course/runs/phase2_lner/bio_bilstm_crf/mha_pre/submission_best/prediction.json
Submission ZIP: /content/drive/MyDrive/legal-llm-course/runs/phase2_lner/bio_bilstm_crf/mha_pre/submission_best/prediction.zip
